# Apply scoring — idempotent rescore + re-rep loop

**This notebook never fires `pixi`/`kubectl` itself.** It reads the already-aggregated
`analysis/aggregated.csv`, decides what needs doing, writes the matrices, and prints the
exact `bash` commands for you to run in a terminal. Then you re-run this notebook and repeat.

Each pass does two things:

1. **Rescore** the `no_scorer` cells that were *killed mid-step by the deadline* (`RUNTIME_KILL`)
   — the scorer re-runs against the saved partial workspace in place (cheap, no agent), turning
   the timeout into an honest `did_run=False` row. See [EXPERIMENTAL_DESIGN.md §Scoring](../EXPERIMENTAL_DESIGN.md#scoring).
2. **Re-rep** the remainder: bring every (model, target) up to `TARGET_N` reps (default = the
   max reps any combo has reached) by launching *fresh* agent cells for the deficit. `THROTTLE`
   / `INFRA_NONSTART` `no_scorer` cells (provider 429 / agent-never-started) aren't rescored —
   they produced no scorable work, so they're simply replaced by the deficit re-rep.

Idempotent by design: if a re-rep wave hits more throttles or timeouts, just re-aggregate and
re-run this notebook — it recomputes from scratch each time. Lather, rinse, repeat until the
deficit is 0 and no `RUNTIME_KILL` remain.

In [ ]:
from pathlib import Path
import sys, csv
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'orchestration'))
sys.path.insert(0, str(REPO_ROOT / 'analysis'))
# Reuse the canonical classifier + cell-id parser so this notebook never drifts
# from orchestration/classify_no_scorer.py.
from classify_no_scorer import classify, PANEL_MODELS
from aggregate import _parse_cell_id  # noqa: F401  (PANEL_MODELS/classify cover our needs)

# === edit per loop =========================================================
AGGREGATED   = REPO_ROOT / 'analysis' / 'aggregated.csv'
# Every batch tag that belongs to the panel. APPEND new rescore-/fill- tags
# here as you launch them, then re-aggregate against the full list.
BATCH_FILTER = [
    'headline-stage1-20260601',
    'headline-rerep-202606020525',
    'headline-rerep-202606021512',
    'fill-*',   # glob: every re-rep wave batch (fill-<ts>); set once, never edit
]
TARGETS      = ['mesa', 'josh', 'josh-mcp']
# None = TARGET_N is the max reps any combo has reached (self-adjusting).
# Pin to an int (e.g. 8) to freeze the target and guarantee convergence.
TARGET_N     = None
# Cap fresh cells emitted per pass. None = fire the WHOLE deficit at once
# (current strategy: launch all, then pick up any throttle/timeout stragglers
# on the next idempotent pass). Set to an int (e.g. 15) to cap each wave —
# round-robin (rep-major) ordering spreads a capped wave across many combos
# rather than hammering one. 75-at-once is well under the 135 that stormed.
WAVE_SIZE    = None
IMAGE_SCORER = 'ghcr.io/schmidtdse/josh-llm-experiment/fortree-scorer:a5b3eae'
IMAGE_AGENT  = 'ghcr.io/schmidtdse/josh-llm-experiment/fortree-agent:a5b3eae'
# ===========================================================================

MODELS = sorted(PANEL_MODELS)
IDX = pd.MultiIndex.from_product([MODELS, TARGETS], names=['model', 'target'])
print(f'panel: {len(MODELS)} models x {len(TARGETS)} targets = {len(MODELS)*len(TARGETS)} combos')
print(f'batches: {BATCH_FILTER}')

In [ ]:
# --- engaged (scorer.json present) per combo -------------------------------
import fnmatch
df = pd.read_csv(AGGREGATED)
# BATCH_FILTER entries may be exact tags OR globs (e.g. 'fill-*'); expand
# against the batch_tag column so the wave loop needs no per-pass edits.
_tags = df.batch_tag.unique().tolist()
_sel = sorted({t for pat in BATCH_FILTER for t in _tags if fnmatch.fnmatch(t, pat)})
df = df[df.batch_tag.isin(_sel) & df.model.isin(PANEL_MODELS) & df.target.isin(TARGETS)].copy()
print(f'matched {len(_sel)} batch tag(s): {_sel}')
df['engaged'] = df.engagement_status != 'no_scorer'
engaged = (df[df.engaged].groupby(['model', 'target']).size()
             .reindex(IDX, fill_value=0).astype(int))
print(f'rows: {len(df)}  |  engaged: {int(engaged.sum())}  |  no_scorer: {int((~df.engaged).sum())}')
print()
print('engaged reps per combo:')
engaged.unstack(fill_value=0).reindex(index=MODELS, columns=TARGETS)

In [ ]:
# --- classify the no_scorer cells from their saved runs/ artifacts ----------
rescore_rows, rerun_dead, review_rows = [], [], []
for _, r in df[~df.engaged].iterrows():
    cell_dir = REPO_ROOT / 'runs' / r.batch_tag / r.run_id
    label, _info = classify(cell_dir) if cell_dir.is_dir() else ('MISSING', {})
    rec = dict(batch_tag=r.batch_tag, run_id=r.run_id, model=r.model, target=r.target, label=label)
    if label == 'RUNTIME_KILL':
        rescore_rows.append(rec)
    elif label in ('THROTTLE', 'INFRA_NONSTART', 'MISSING'):
        rerun_dead.append(rec)
    else:
        review_rows.append(rec)

print(f'no_scorer breakdown:')
print(f'  RUNTIME_KILL  {len(rescore_rows):>3}  -> rescore in place (did_run=False)')
print(f'  THROTTLE/INFRA{len(rerun_dead):>3}  -> no scorable work; replaced by the deficit re-rep')
print(f'  REVIEW        {len(review_rows):>3}  -> hand-inspect (should be 0)')
for rec in review_rows:
    print(f'      REVIEW: {rec["batch_tag"]}/{rec["run_id"]}')

In [ ]:
# --- write the rescore manifest (RUNTIME_KILL cells) ------------------------
rescore_manifest = REPO_ROOT / 'orchestration' / 'rescore-manifest.csv'
with rescore_manifest.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['orig_batch_tag', 'run_id', 'target', 'minio_prefix'])
    w.writeheader()
    for rec in rescore_rows:
        # minio_prefix == batch_tag (how the originals were rendered:
        # render_jobs.py uses `minio_prefix or batch_tag`).
        w.writerow(dict(orig_batch_tag=rec['batch_tag'], run_id=rec['run_id'],
                        target=rec['target'], minio_prefix=rec['batch_tag']))
print(f'wrote {len(rescore_rows)} cell(s) -> {rescore_manifest.relative_to(REPO_ROOT)}')

In [ ]:
# --- compute the deficit & write the re-rep matrix --------------------------
# Projected engaged = current engaged + the RUNTIME_KILL cells that the rescore
# will turn into counted rows. THROTTLE/INFRA are NOT added (no scorable work).
rescorable = pd.Series(0, index=IDX, dtype=int)
for rec in rescore_rows:
    rescorable[(rec['model'], rec['target'])] += 1
projected = engaged + rescorable

N = int(projected.max()) if TARGET_N is None else int(TARGET_N)
deficit = (N - projected).clip(lower=0).astype(int)

rerun_matrix = REPO_ROOT / 'orchestration' / 'matrix-rescore-rerun.csv'
# Round-robin (rep-major): rep 0 of every deficient combo, then rep 1, etc., so
# a WAVE_SIZE-capped slice touches many combos one rep each instead of all reps
# of a single combo.
max_def = int(deficit.max()) if len(deficit) else 0
all_rows = [(m, t) for rep in range(max_def)
            for (m, t), n in deficit.items() if rep < int(n)]
total_deficit = len(all_rows)
rows = all_rows[:int(WAVE_SIZE)] if WAVE_SIZE is not None else all_rows
with rerun_matrix.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['model', 'target']); w.writerows(rows)

print(f'TARGET_N = {N}  ({"max projected reps" if TARGET_N is None else "pinned"})')
print(f'total deficit to N = {total_deficit} fresh cell(s)')
capped = WAVE_SIZE is not None and total_deficit > len(rows)
print(f'THIS WAVE = {len(rows)} cell(s)' + (f'  (capped at WAVE_SIZE={WAVE_SIZE}; '
      f'{total_deficit - len(rows)} left for later waves)' if capped else '')
      + f'  -> {rerun_matrix.relative_to(REPO_ROOT)}')
print()
print('deficit per combo (nonzero):')
print(deficit[deficit > 0].reset_index(name='reps').to_string(index=False) if int(deficit.sum()) else '  (none)')

In [ ]:
# --- convergence check + ready-to-paste commands ----------------------------
done = (int(deficit.sum()) == 0) and (len(rescore_rows) == 0)
print('=' * 70)
if done:
    print(f'✓ CONVERGED: every combo at N={N}, no RUNTIME_KILL pending. '
          f'Re-run analysis/01_analysis.ipynb for the final figures.')
else:
    ts = '$(date -u +%Y%m%d%H%M)'
    pulls = ' '.join(BATCH_FILTER)
    if rescore_rows:
        print(f'▶ STEP 1 — rescore {len(rescore_rows)} RUNTIME_KILL cell(s) in place:')
        print(f'''
  pixi run rescore -- --batch-tag rescore-{ts} \\
      --manifest orchestration/rescore-manifest.csv \\
      --image-scorer {IMAGE_SCORER} --watch
  for b in {pulls}; do MINIO_PREFIX=$b pixi run pull $b; done
''')
    if len(rows):
        print(f'▶ STEP 2 — launch {len(rows)} fresh agent cell(s) (use a SMALL wave / low '
              f'concurrency to avoid the OpenRouter throttle storm):')
        print(f'''
  FILL=fill-{ts}
  pixi run apply -- --batch-tag $FILL \\
      --image-agent {IMAGE_AGENT} --image-scorer {IMAGE_SCORER} \\
      --matrix orchestration/matrix-rescore-rerun.csv
  pixi run pull $FILL
''')
    print('▶ STEP 3 — re-aggregate ALL tags (append the new rescore-/fill- tags to')
    print('  BATCH_FILTER above first), then RE-RUN THIS NOTEBOOK. Repeat until converged.')
    print(f'''
  pixi run aggregate runs/{pulls.replace(' ', ' runs/')} runs/rescore-{ts} runs/fill-{ts}
''')
print('=' * 70)